# Phase 0 smoke test
Runs a PySpark job on the cluster (`spark://spark-master:7077`) that writes to and reads from HDFS, checks driver/executor Python versions match, then checks Neo4j + GDS from the same container.

Run headless with `make test-phase0`.

In [ ]:
import os, sys
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("phase0-smoke")
         .master(os.environ.get("SPARK_MASTER_URL", "spark://spark-master:7077"))
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| master", sc.master, "| driver Python", sys.version.split()[0])

In [ ]:
path = os.environ.get("HDFS_URL", "hdfs://namenode:9000") + "/test/phase0_smoke"

df = spark.range(0, 100_000).withColumn("square", F.col("id") * F.col("id"))
df.write.mode("overwrite").parquet(path)

back = spark.read.parquet(path)
n, total = back.count(), back.agg(F.sum("square")).first()[0]
assert n == 100_000, n
print(f"HDFS write/read OK: {path} rows={n} sum(square)={total}")

In [ ]:
# A Python UDF runs on the executors: it fails if worker and driver Python differ.
@F.udf("string")
def worker_info(_):
    import platform, socket
    return f"{socket.gethostname()} py{platform.python_version()}"

workers = sorted(r[0] for r in spark.range(0, 64, numPartitions=8).select(worker_info("id")).distinct().collect())
print("Executors seen:", workers)
assert workers, "no executor output"
spark.stop()
print("SMOKE_SPARK_HDFS_OK")

In [ ]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(os.environ.get("NEO4J_URI", "bolt://neo4j:7687"),
                       auth=(os.environ.get("NEO4J_USER", "neo4j"), os.environ["NEO4J_PASSWORD"]))
print("GDS version:", gds.server_version())
gds.close()
print("SMOKE_NEO4J_GDS_OK")